In [ ]:
!pip install "psycopg[binary]==3.3.6" "python-dotenv==1.2.4"

This installs the pinned Python packages: `psycopg` (PostgreSQL driver) and `python-dotenv` (loads `.env`).

In [ ]:
# Imports and helpers
import os
from decimal import Decimal

import psycopg
from psycopg import sql
from dotenv import load_dotenv

We import the standard library helpers, `psycopg`, the `sql` builder, and `load_dotenv`.

In [ ]:
# Load environment
load_dotenv("../../.env")
PGHOST = os.getenv("PGHOST", "localhost")
PGPORT = os.getenv("PGPORT", "5432")
PGUSER = os.getenv("PGUSER", "postgres")
PGPASSWORD = os.getenv("PGPASSWORD") or None
PGDATABASE = os.getenv("PGDATABASE", "university")
PGSSLMODE = os.getenv("PGSSLMODE", "prefer")
print(f"Connecting to {PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE} (sslmode={PGSSLMODE})")

Loads connection settings from the shared `.env` at the course root.

In [ ]:
# Ensure database exists
conn_args = dict(host=PGHOST, port=PGPORT, user=PGUSER, password=PGPASSWORD, sslmode=PGSSLMODE)
with psycopg.connect(dbname="postgres", autocommit=True, **conn_args) as admin:
    exists = admin.execute("SELECT 1 FROM pg_database WHERE datname=%s", (PGDATABASE,)).fetchone()
    if not exists:
        admin.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(PGDATABASE)))
print("db ready")

Ensures the `university` database exists (self-contained).

In [ ]:
# Connect (no autocommit)
conn = psycopg.connect(dbname=PGDATABASE, **conn_args)
print("connected")

Opens a connection with autocommit off so we can commit explicitly.

In [ ]:
# show helper
def show(query, params=None):
    """Run a query and print the result as a psql-style table."""
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description is None:              # statements like CREATE or INSERT return no rows
            print(cur.statusmessage)
            return
        cols = [d.name for d in cur.description]
        rows = cur.fetchall()
    fmt = lambda v: "NULL" if v is None else ("t" if v is True else "f" if v is False else str(v))
    cells = [[fmt(v) for v in r] for r in rows]
    numeric = [bool(rows) and all(v is None or (isinstance(v, (int, float, Decimal)) and not isinstance(v, bool)) for v in col)
               for col in zip(*rows)] or [False] * len(cols)
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    print(" | ".join(c.center(w) for c, w in zip(cols, widths)).rstrip())
    print("-+-".join("-" * w for w in widths))
    for r in cells:
        print(" | ".join(v.rjust(w) if numeric[i] else v.ljust(w) for i, (v, w) in enumerate(zip(r, widths))).rstrip())
    print(f"({len(rows)} row{'' if len(rows) == 1 else 's'})")
show("SELECT current_database()")

A small helper to print aligned results (similar to lesson 2).

In [ ]:
# Step 1: Model tables with foreign keys
conn.execute("DROP TABLE IF EXISTS enrollments CASCADE")
conn.execute("DROP TABLE IF EXISTS courses CASCADE")
conn.execute("DROP TABLE IF EXISTS students CASCADE")

conn.execute("""CREATE TABLE students (
    student_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE
)""")
conn.execute("""CREATE TABLE courses (
    course_id TEXT PRIMARY KEY,
    code TEXT NOT NULL UNIQUE,
    title TEXT NOT NULL,
    credits INTEGER NOT NULL CHECK (credits > 0)
)""")
conn.execute("""CREATE TABLE enrollments (
    enrollment_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    student_id TEXT NOT NULL REFERENCES students(student_id) ON DELETE RESTRICT,
    course_id TEXT NOT NULL REFERENCES courses(course_id) ON DELETE RESTRICT,
    grade INTEGER CHECK (grade BETWEEN 0 AND 100),
    UNIQUE (student_id, course_id)
)""")
conn.commit()
print("tables created")

Creates normalized tables with primary keys, foreign keys (`ON DELETE RESTRICT`), `UNIQUE`, and `CHECK`. The junction `enrollments` represents a many-to-many between students and courses. Foreign keys are not automatically indexed in PostgreSQL — add indexes on the referencing columns if you need them for performance.

In [ ]:
# Step 1b: foreign keys in action. First load one valid student, course and enrollment.
conn.execute("INSERT INTO students VALUES ('STU001','Alice Johnson','alice@univ.example')")
conn.execute("INSERT INTO courses VALUES ('C001','CS101','Intro to CS',3)")
conn.execute("INSERT INTO enrollments (student_id,course_id,grade) VALUES ('STU001','C001',95)")
conn.commit()

# 1) Enroll a student who does not exist: the foreign key refuses it.
try:
    conn.execute("INSERT INTO enrollments (student_id, course_id) VALUES ('NOPE', 'C001')")
    conn.commit()
except psycopg.errors.IntegrityError as err:
    conn.rollback()
    print("Refused:", err.diag.message_primary)

# 2) Delete a course that still has enrollments: ON DELETE RESTRICT refuses it.
try:
    conn.execute("DELETE FROM courses WHERE course_id = 'C001'")
    conn.commit()
except psycopg.errors.IntegrityError as err:
    conn.rollback()
    print("Refused:", err.diag.message_primary)

Library-card analogy: you cannot enroll a student who does not exist (the first statement is refused), and you cannot delete a course that still has enrollments (the second is refused) because the foreign keys use `ON DELETE RESTRICT`. A refused insert still uses up an identity number, so the `enrollment_id` values below can skip a number; that is normal.

In [ ]:
# Step 2: insert the rest of the sample data (STU004 has no enrollment, ENG101 has no students)
conn.execute("INSERT INTO students VALUES ('STU002','Bob Smith','bob@univ.example'),('STU003','Charlie Brown','charlie@univ.example'),('STU004','Dana Lee',NULL)")
conn.execute("INSERT INTO courses VALUES ('C002','MATH101','Calculus I',4),('C003','ENG101','Writing',3)")
conn.execute("INSERT INTO enrollments (student_id,course_id,grade) VALUES ('STU001','C002',88),('STU002','C001',72),('STU003','C002',91)")
conn.commit()
show("SELECT * FROM students ORDER BY student_id")
show("SELECT * FROM courses ORDER BY course_id")
show("SELECT * FROM enrollments ORDER BY enrollment_id")

Loads the rest of the sample data. STU004 (Dana) has no enrollments and ENG101 has no students, so the outer joins will differ. The junction table links students and courses.

In [ ]:
# Step 3: INNER JOIN — only matches
show("""
  SELECT s.student_id, s.name, c.code, c.title, e.grade
  FROM enrollments e
  INNER JOIN students s ON s.student_id = e.student_id
  INNER JOIN courses c ON c.course_id = e.course_id
  ORDER BY s.student_id, c.code
""")

`INNER JOIN` returns only rows that match in all joined tables.

In [ ]:
# Step 4: LEFT JOIN — all students, even those with no enrollment
show("""
  SELECT s.student_id, s.name, c.code, e.grade
  FROM students s
  LEFT JOIN enrollments e ON e.student_id = s.student_id
  LEFT JOIN courses c ON c.course_id = e.course_id
  ORDER BY s.student_id, c.code
""")

`LEFT JOIN` keeps all rows from the left table (`students`); unmatched right-side values become `NULL`. Dana Lee (STU004) appears with NULLs because she has no enrollments.

In [ ]:
# Step 5: RIGHT JOIN and FULL OUTER JOIN
# RIGHT JOIN: keep every course, even one nobody takes
show("""
  SELECT c.code, e.student_id
  FROM enrollments e
  RIGHT JOIN courses c ON c.course_id = e.course_id
  ORDER BY c.code, e.student_id
""")
# FULL OUTER JOIN: keep unmatched rows from both sides (Dana has no course, ENG101 has no student)
show("""
  SELECT s.name, c.code
  FROM students s
  FULL OUTER JOIN enrollments e ON e.student_id = s.student_id
  FULL OUTER JOIN courses c ON c.course_id = e.course_id
  ORDER BY s.name NULLS LAST, c.code NULLS LAST
""")

`RIGHT JOIN` keeps all rows of the right-hand table (here every course, so ENG101 appears with a NULL student). `FULL OUTER JOIN` keeps unmatched rows from both sides at once: Dana Lee (no course) and ENG101 (no student).

In [ ]:
# Step 6: CROSS JOIN
show("""
  SELECT s.name, c.code
  FROM students s CROSS JOIN courses c
  ORDER BY s.name, c.code
""")

`CROSS JOIN` produces the Cartesian product (all pairs). Useful to see possible combinations.

In [ ]:
# Step 7: SELF JOIN (courses could relate via prerequisites)
conn.execute("ALTER TABLE courses ADD COLUMN IF NOT EXISTS prerequisite TEXT REFERENCES courses(course_id)")
conn.execute("UPDATE courses SET prerequisite='C001' WHERE course_id='C002'")
conn.commit()
show("""
  SELECT c.code AS course, p.code AS prereq
  FROM courses c
  LEFT JOIN courses p ON p.course_id = c.prerequisite
  ORDER BY c.code
""")

A `SELF JOIN` joins a table to itself to represent hierarchical relationships like prerequisites.

In [ ]:
# Step 8: UNION, INTERSECT, EXCEPT
show("""
  SELECT student_id FROM enrollments WHERE course_id='C001'
  UNION
  SELECT student_id FROM enrollments WHERE course_id='C002'
  ORDER BY student_id
""")
show("""
  SELECT student_id FROM enrollments WHERE course_id='C001'
  INTERSECT
  SELECT student_id FROM enrollments WHERE course_id='C002'
  ORDER BY student_id
""")
show("""
  SELECT student_id FROM enrollments WHERE course_id='C001'
  EXCEPT
  SELECT student_id FROM enrollments WHERE course_id='C002'
  ORDER BY student_id
""")


Set operations combine result sets: `UNION` (distinct), `INTERSECT` (common), `EXCEPT` (difference).

In [ ]:
# Step 9: showcase a combined report
show("""
  SELECT s.student_id, s.name, string_agg(c.code, ', ' ORDER BY c.code) AS courses, ROUND(avg(e.grade), 1) AS avg_grade
  FROM students s
  LEFT JOIN enrollments e ON e.student_id = s.student_id
  LEFT JOIN courses c ON c.course_id = e.course_id
  GROUP BY s.student_id, s.name
  ORDER BY s.student_id
""")

A small report that brings relationships together (joins + aggregation; `GROUP BY` and `avg` are explained fully later in the course).

In [ ]:
# Cleanup
conn.close()
print("connection closed")

Closes the connection. The tables and rules remain in the database.